# Comparison of perm2code Implementations

Comparision between `perm2code` and `perm2code_2`.

In [1]:
import time

import numpy as np

from lehmer import Lehmer

In [ ]:
def compare_functions(
    lc: Lehmer,
    test_vector,
    name1: str = "Function 1",
    name2: str = "Function 2",
    number: int = 100,
    repeat: int = 3,
) -> dict:

    # Warmup
    lc.perm2code(test_vector)
    lc.perm2code_2(test_vector)

    times1 = []
    for _ in range(repeat):
        start = time.perf_counter()
        for _ in range(number):
            lc.perm2code(test_vector)
        end = time.perf_counter()
        times1.append((end - start) / number)

    times2 = []
    for _ in range(repeat):
        start = time.perf_counter()
        for _ in range(number):
            lc.perm2code_2(test_vector)
        end = time.perf_counter()
        times2.append((end - start) / number)

    mean1 = np.mean(times1)
    mean2 = np.mean(times2)
    return mean1.item(), mean2.item()


In [3]:
def compare(n, batch, number=1000, repeat=5):
    lc = Lehmer(n=n)
    test_vector = [np.random.permutation(n) for _ in range(batch)]
    perm2code, perm2code_2 = compare_functions(lc, test_vector, number=number, repeat=repeat)
    return perm2code, perm2code_2

In [4]:
results = {}
batches = [1, 5, 10, 20, 50, 100, 1000]
for n in range(5, 31):
    for batch in batches:
        perm2code_time, perm2code_2_time = compare(n, batch)
        results[(n, batch)] = (perm2code_time, perm2code_2_time)

In [5]:
import pandas as pd

table_data = []
for n in range(5, 31):
    row = {'n': n}
    for batch in batches:
        perm2code_time, perm2code_2_time = results[(n, batch)]
        row[f'b={batch}'] = f'{perm2code_time*1e6/batch:.2f} / {perm2code_2_time*1e6/batch:.2f}'
    table_data.append(row)

df = pd.DataFrame(table_data)

In [6]:
def color_cell(val):
    if '/' not in str(val):
        return ''
    parts = str(val).split(' / ')
    time1 = float(parts[0])
    time2 = float(parts[1])
    if time1 < time2:
        return 'color: green'
    else:
        return 'color: red'


The following table displays the timing results for the two implementations of the `perm2code` function. The results, average time in microseconds,  are shown side by side like this `perm2code/perm2code_2`. Green cells indicate that `perm2code` was faster. `n` is the length of the permutation and `b` is the batch size (number of permutations processed simultaneously).

In [7]:

df.style.applymap(color_cell, subset=[col for col in df.columns if col != 'n'])

/var/folders/ml/lm19ddxj2tg9_zt3rj4z3m_00000gn/T/ipykernel_41810/201831833.py:1: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  df.style.applymap(color_cell, subset=[col for col in df.columns if col != 'n'])


,n,b=1,b=5,b=10,b=20,b=50,b=100,b=1000
0,5,9.50 / 13.58,2.13 / 3.04,1.14 / 1.57,0.68 / 0.87,0.42 / 0.48,0.37 / 0.31,0.25 / 0.20
1,6,10.42 / 27.68,2.07 / 3.52,1.17 / 1.90,0.77 / 1.08,0.47 / 0.55,0.37 / 0.39,0.27 / 0.22
2,7,9.88 / 18.91,2.16 / 4.12,1.22 / 2.18,0.77 / 1.19,0.49 / 0.60,0.41 / 0.41,0.32 / 0.26
3,8,9.41 / 22.89,2.36 / 4.75,1.24 / 2.50,0.78 / 1.43,0.48 / 0.68,0.42 / 0.47,0.30 / 0.27
4,9,9.55 / 24.34,2.22 / 5.54,1.27 / 2.81,0.79 / 1.51,0.51 / 0.79,0.45 / 0.51,0.33 / 0.31
5,10,9.72 / 27.20,2.37 / 6.03,1.30 / 3.28,0.83 / 1.68,0.59 / 0.86,0.49 / 0.56,0.36 / 0.32
6,11,9.91 / 30.15,2.39 / 6.64,1.36 / 3.41,0.87 / 1.85,0.64 / 0.91,0.54 / 0.61,0.42 / 0.34
7,12,9.73 / 32.18,2.33 / 7.06,1.40 / 3.67,0.92 / 1.99,0.67 / 1.02,0.56 / 0.68,0.43 / 0.39
8,13,10.00 / 34.81,2.42 / 7.68,1.46 / 4.01,0.97 / 2.16,0.74 / 1.06,0.60 / 0.70,0.49 / 0.40
9,14,10.12 / 37.50,2.45 / 8.21,1.52 / 4.29,1.02 / 2.36,0.79 / 1.18,0.65 / 0.85,0.54 / 0.43
